# SpikeBit — Stage A on Colab

Runs the Stage A pipeline (teacher PPL → conversion → QAT fine-tune → eval)
on a Colab GPU. Checkpoints, runs and the packed dataset are kept on Google
Drive so a session disconnect does not lose them.

Runtime → Change runtime type → GPU (T4 works; L4/A100 are faster).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.get_device_capability())

In [ ]:
REPO_URL = "https://github.com/Orolol/SpikeBit.git"  # private repo: use a token or `gh auth` first
BRANCH = "main"
import os
if not os.path.isdir("/content/SpikeBit"):
    !git clone --branch $BRANCH $REPO_URL /content/SpikeBit
%cd /content/SpikeBit
!git pull --ff-only

In [ ]:
# Colab ships its own torch; install the project on top of it (no torch reinstall).
!pip install -q -e . 2>&1 | tail -2
!python scripts/check_env.py

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/SpikeBit"
for sub in ("checkpoints", "runs", "data"):
    os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
    if os.path.islink(sub) or not os.path.exists(sub):
        !rm -rf $sub && ln -s $DRIVE/$sub $sub
!ls -la checkpoints runs data

## 1. Teacher reference PPL

Same eval tail as `scripts/evaluate.py`, so the number is comparable with the student. Never measured so far.

In [ ]:
!python scripts/eval_teacher.py --seq-len 128 --batch 2 --max-tokens 20000 2>&1 | tee runs/teacher_ppl.txt | tail -5

## 2. Convert the teacher into the 4-bit spiking-ternary student

`--no-spiking-r` = FP receptance (DECISIONS #9/#11). Skipped if the checkpoint already exists on Drive.

In [ ]:
INIT = "checkpoints/stage_a_init_4b.pt"
if not os.path.exists(INIT):
    !python scripts/convert_rwkv.py --out $INIT --bits 4 --no-spiking-r
else:
    print("exists:", INIT)

## 3. Stage A fine-tune (full 10 k steps)

Fused CUDA kernels (JIT-built for this GPU on first use, ~1–2 min). On a T4 there are no BF16 tensor cores, so use `--mixed-precision none`; on L4/A100 keep `bf16`.
Checkpoints go to Drive every `ckpt_every` steps. The trainer has no optimizer-state resume: if the session dies, restart from the last `step_*.pt` as `--init-checkpoint` (fresh optimizer, shorter warmup).

In [ ]:
cap = torch.cuda.get_device_capability()
MP = "bf16" if cap >= (8, 0) else "none"
RUN = "stage_a_full_4b"
!python scripts/train_stage_a.py \
    --init-checkpoint $INIT \
    --out-dir checkpoints/$RUN --metrics-path runs/$RUN/metrics.jsonl \
    --seq-len 64 --batch-size 8 --grad-accum 1 \
    --total-steps 10000 --warmup-steps 500 \
    --lr-embed-head 1e-4 \
    --mixed-precision $MP \
    --log-every 50 --eval-every 500 --ckpt-every 1000 2>&1 | tail -20

## 4. Evaluate the student

In [ ]:
!python scripts/evaluate.py --checkpoint checkpoints/$RUN/final.pt 2>&1 | tail -30

In [ ]:
# Loss / eval-PPL curve from the JSONL metrics.
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f"runs/{RUN}/metrics.jsonl")]
tr = [(r["step"], r["loss"]) for r in rows if "loss" in r]
ev = [(r["step"], r["eval_ppl"]) for r in rows if "eval_ppl" in r]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(*zip(*tr)); ax[0].set_yscale("log"); ax[0].set_title("train loss")
if ev: ax[1].plot(*zip(*ev), marker="o"); ax[1].set_yscale("log"); ax[1].set_title("eval PPL")
plt.show()